# 5b. MBAR Weighted Resampling from T-REMD

This tutorial takes a **Temperature Replica-Exchange Molecular Dynamics (T-REMD)** dataset through a complete MBAR workflow. We reweight samples to a target temperature, draw frames with replacement according to their MBAR weights, and produce a conventional trajectory whose frames all carry uniform weight.

The resampled trajectory can be analyzed like an ordinary MD trajectory—for example, with unweighted histograms, clustering tools, or database-deposition pipelines. This makes enhanced-sampling results easier to use in downstream workflows.

## Workflow

1. **Inspect the dataset** — trialanine (Ala3; 42 solute atoms), 20 temperatures from 300.00 to 351.26 K, and 5,000 frames per state
2. **Run standard MBAR** — estimate the relative free energy `fene` of every temperature state and compare it with the reference result
3. **Inspect per-sample weights** — use `return_weights=True` to obtain the weight of every sample in the unbiased 300 K target ensemble
4. **Build a reweighted Ramachandran PMF** — weight the φ/ψ samples to reconstruct the 300 K free-energy surface
5. **Resample the trajectory** — draw frames with replacement and verify that an ordinary, unweighted φ/ψ histogram reproduces the weighted PMF
6. **Save and reuse the result** — write the uniform-weight ensemble to DCD for downstream analysis

> The tutorial dataset is downloaded from Google Drive on demand with `gdown`; it is not stored in the repository.

## 0. Setup: download the dataset

Download the complete T-REMD tutorial dataset (about 50 MB). The download runs only once; subsequent executions reuse the extracted files.

In [ ]:
from genepie.tests import download_tremd_data

rc = download_tremd_data.download()
assert rc == 0 and download_tremd_data.is_available(), \
    "T-REMD data unavailable. Ensure gdown is installed: pip install gdown"
print("T-REMD tutorial data is ready.")

## 1. Inspect the dataset

We use a T-REMD simulation of **trialanine (Ala3)**. The raw replica trajectories have already been sorted by temperature (parameter ID) with `remd_convert`, producing one energy series and one trajectory for each temperature state.

| File | Contents |
|------|----------|
| `remd_paramID{k}.pot` | Potential-energy time series (`time value`) for state `k`; used as the MBAR `cvfile` |
| `remd_paramID{k}_trialanine.dcd` | 42-atom solute trajectory for state `k` |
| `trialanine.pdb` / `ala3.psf` | Molecular topology |
| `reference/` | Published MBAR reference results: `fene.dat`, `weight{k}.dat`, and `remd_paramID{k}.tor` |

The energy samples and trajectory frames must be aligned **one to one**. Every `.pot` file and DCD in this dataset contains exactly 5,000 samples.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from genepie.tests import conftest as C

NREPLICA = C.TREMD_NREPLICA
TEMPERATURES = C.TREMD_TEMPERATURES
TARGET_T = C.TREMD_TARGET_TEMPERATURE

print(f"replicas        : {NREPLICA}")
print(f"temperatures (K): {TEMPERATURES[0]:.2f} ... {TEMPERATURES[-1]:.2f}")
print(f"target T (K)    : {TARGET_T}")
print(f"cvfile pattern  : {C.TREMD_POT_PATTERN}")
print(f"dcd pattern     : {C.TREMD_DCD_PATTERN}")

# One state's potential-energy series (samples per state).
pot0 = np.loadtxt(C.TREMD_POT_PATTERN.format(1))
n_step = pot0.shape[0]
print(f"\nsamples per state: {n_step}  (total = {n_step * NREPLICA})")

In [ ]:
from genepie.s_molecule import SMolecule

mol = SMolecule.from_file(pdb=str(C.TREMD_PDB), psf=str(C.TREMD_PSF), ref=str(C.TREMD_PDB))
print(f"solute atoms: {mol.num_atoms}")

## 2. Run standard MBAR

MBAR combines samples collected at different temperatures (thermodynamic states) to estimate the relative free energy of every state. For T-REMD, pass `input_type="EneSingle"` (equivalent to `REMD` inside GENESIS), the temperature ladder, and the desired `target_temperature`.

> The Fortran MBAR solver retains global state between calls. Use `mbar_analysis_isolated`, which performs each solve in a clean subprocess and is safe for repeated analyses.

In [ ]:
from genepie import genesis_exe

result = genesis_exe.mbar_analysis_isolated(
    cvfile=C.TREMD_POT_PATTERN,
    nreplica=NREPLICA,
    input_type="EneSingle",
    dimension=1,
    temperature=TEMPERATURES,
    target_temperature=TARGET_T,
    tolerance=1e-8,
    self_iteration=100,
    newton_iteration=10,
    return_weights=True,   # Return per-sample weights together with fene
    timeout=600.0,
)

fene = result.fene[:, 0]   # shape (nreplica,)
print("relative free energy per state (kcal/mol):")
print(fene)

In [ ]:
# Compare with the paper reference fene.dat.
fene_ref = np.loadtxt(str(C.TREMD_FENE_REF))
max_abs_err = np.max(np.abs(fene - fene_ref))
print(f"max |fene - reference| = {max_abs_err:.2e}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(TEMPERATURES, fene, "o-", label="genepie")
ax.plot(TEMPERATURES, fene_ref, "x--", label="reference")
ax.set_xlabel("temperature (K)")
ax.set_ylabel("relative free energy (kcal/mol)")
ax.set_title("MBAR free energy per state")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Understand the per-sample weights

With `return_weights=True`, MBAR returns the contribution of every sample to the **unbiased 300 K target ensemble** as `w[state][frame]`. The array has shape `(n_replica, n_step)`, and all entries together sum to one.

Samples that are representative of the 300 K ensemble receive larger weights. Samples drawn mainly because a replica was hot—particularly high-energy structures that are rare at 300 K—receive smaller weights. The weights therefore combine information from every temperature while correcting each sample's contribution to the target ensemble.

In [ ]:
weights = result.weights   # (n_replica, n_step)
print(f"weights shape : {weights.shape}")
print(f"total sum     : {weights.sum():.6f}  (should be ~1.0)")

# Weight mass carried by each temperature state.
state_mass = weights.sum(axis=1)
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(range(1, NREPLICA + 1), state_mass)
ax.set_xlabel("state index (cold -> hot)")
ax.set_ylabel("total weight at target T")
ax.set_title(f"Weight concentrates in cold states (target {TARGET_T} K)")
plt.tight_layout()
plt.show()

In [ ]:
# The weights match the paper reference weight{k}.dat.
ref_w = np.stack([np.loadtxt(C.TREMD_WEIGHT_PATTERN.format(k))[:, 1]
                  for k in range(1, NREPLICA + 1)])
print(f"max |weight - reference| = {np.max(np.abs(weights - ref_w)):.2e}")

## 4. Build a reweighted Ramachandran PMF

We now reconstruct the 300 K free-energy surface from the trialanine backbone dihedrals φ and ψ. The example uses the reference `.tor` files supplied with the dataset; the same angles can also be calculated with `trj_analysis`.

Each `.tor` file contains `index  phi  psi`. Concatenating the files in `(state, step)` order gives the same ordering as the flattened MBAR weight array, so a weighted two-dimensional histogram directly estimates the target-ensemble probability and PMF.

In [ ]:
phi_states, psi_states = [], []
for k in range(1, NREPLICA + 1):
    tor = np.loadtxt(C.TREMD_TOR_PATTERN.format(k))
    phi_states.append(tor[:, 1])
    psi_states.append(tor[:, 2])
phi = np.concatenate(phi_states)   # (N,)
psi = np.concatenate(psi_states)   # (N,)
w_flat = weights.ravel()
w_flat = w_flat / w_flat.sum()
print(f"total samples: {phi.size}")

In [ ]:
KB = 0.0019872041  # kcal/mol/K
bins = np.linspace(-180, 180, 73)

def pmf_2d(phi, psi, weights=None):
    H, xe, ye = np.histogram2d(phi, psi, bins=[bins, bins], weights=weights)
    P = H / H.sum()
    with np.errstate(divide="ignore"):
        F = -KB * TARGET_T * np.log(P)
    F -= np.nanmin(F[np.isfinite(F)])
    return np.ma.masked_invalid(F), xe, ye

# Naive (unweighted) histogram vs weight-reweighted PMF.
F_naive, xe, ye = pmf_2d(phi, psi)
F_reweighted, _, _ = pmf_2d(phi, psi, weights=w_flat)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, F, title in [(axes[0], F_naive, "naive (all states pooled)"),
                     (axes[1], F_reweighted, f"MBAR-reweighted ({TARGET_T} K)")]:
    pc = ax.pcolormesh(xe, ye, F.T, cmap="viridis", vmin=0, vmax=6)
    ax.set_xlabel("phi (deg)"); ax.set_ylabel("psi (deg)"); ax.set_title(title)
    fig.colorbar(pc, ax=ax, label="free energy (kcal/mol)")
plt.tight_layout()
plt.show()

## 5. Resample the trajectory

Drawing frames **with replacement** according to the MBAR weights converts the weighted sample set into a trajectory whose frames all carry uniform weight. `mbar_resample_trajectory` performs the complete pipeline: solve MBAR, load the per-state DCD files, draw frame indices, construct the resampled trajectory, and optionally save it as DCD.

In [ ]:
import os, tempfile

dcd_files = [C.TREMD_DCD_PATTERN.format(k) for k in range(1, NREPLICA + 1)]
out_dir = tempfile.mkdtemp(prefix="mbar_resample_")
out_dcd = os.path.join(out_dir, "resampled_300K.dcd")

res = genesis_exe.mbar_resample_trajectory(
    molecule=mol,
    dcd_files=dcd_files,
    cvfile=C.TREMD_POT_PATTERN,
    nreplica=NREPLICA,
    temperature=TEMPERATURES,
    target_temperature=TARGET_T,
    input_type="EneSingle",
    self_iteration=100,
    newton_iteration=10,
    selection="all",       # The DCD files already contain only the solute
    n_samples=phi.size,    # Draw as many frames as the original sample count
    seed=2024,
    output_dcd=out_dcd,
    isolated=True,
)

print(f"resampled frames: {res.trajectory.nframe}")
print(f"atoms per frame : {res.trajectory.natom}")
print(f"saved DCD       : {out_dcd}  ({os.path.getsize(out_dcd)/1e6:.2f} MB)")

In [ ]:
# The resampled ensemble's PLAIN histogram must reproduce the reweighted PMF.
# res.indices are indices into the same (state, step) order as phi/psi.
idx = res.indices
F_resampled, _, _ = pmf_2d(phi[idx], psi[idx])   # no weights: uniform now

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, F, title in [(axes[0], F_reweighted, "MBAR-reweighted (weights)"),
                     (axes[1], F_resampled, "resampled (uniform weight)")]:
    pc = ax.pcolormesh(xe, ye, F.T, cmap="viridis", vmin=0, vmax=6)
    ax.set_xlabel("phi (deg)"); ax.set_ylabel("psi (deg)"); ax.set_title(title)
    fig.colorbar(pc, ax=ax, label="free energy (kcal/mol)")
plt.tight_layout()
plt.show()

# Quantitative check on the 1-D phi distribution.
coarse = np.linspace(-180, 180, 19)
h_rw, _ = np.histogram(phi, bins=coarse, weights=w_flat)
h_rs, _ = np.histogram(phi[idx], bins=coarse); h_rs = h_rs / h_rs.sum()
print(f"max |reweighted - resampled| (phi hist) = {np.max(np.abs(h_rw - h_rs)):.4f}")

In [ ]:
# Which states did the draws come from? Cold states dominate at 300 K.
state_of_draw = res.indices // res.weights.shape[1]
counts = np.bincount(state_of_draw, minlength=NREPLICA)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(range(1, NREPLICA + 1), counts)
ax.set_xlabel("source state index (cold -> hot)")
ax.set_ylabel("# resampled frames")
ax.set_title("Provenance of resampled frames")
plt.tight_layout()
plt.show()

## 6. Save and reuse the ensemble

The generated DCD (`resampled_300K.dcd`) and the `trialanine.pdb` topology form a conventional **single-ensemble trajectory**. It can be passed directly to MDTraj, MDAnalysis, clustering workflows, or deposition pipelines such as MDDB without carrying a separate weight for every frame.

**Summary**
- MBAR converts samples from multiple thermodynamic states into per-sample weights for an unbiased target ensemble.
- Sampling frames with replacement according to those weights produces a single trajectory with uniform frame weights.
- The ordinary histogram of the resampled frames reproduces the weighted PMF, providing a direct validation of the resampling procedure.

In [ ]:
import mdtraj

t = mdtraj.load(out_dcd, top=str(C.TREMD_PDB))
print(t)